# RTX4090 local runner for updated shrimp classification scripts

This notebook pulls the latest branch, validates the local Python/CUDA environment, uses the local dataset at `processed-images/processed_images`, prepares manifests, and runs the updated experiment scripts with strict resume/skip behavior.

No smoke-test stages are included. Real run switches default to `True`; completed runs are skipped only by the scripts' strict final-metric validation. Destructive overwrite flags default to `False`.

This notebook does not store a GitHub token in the file. Set `GITHUB_TOKEN` in the environment or paste it into the hidden prompt when asked.

In [4]:
import os
import shlex
import shutil
import subprocess
import time
import getpass
from pathlib import Path
from datetime import datetime, timezone

WORKDIR = Path("/home/drnguyenvinh/notebooks").expanduser().resolve()
REPO_DIR = WORKDIR / "CVio_Shrimp_Disease_Classification_Capstone_SU26"
LOG_DIR = WORKDIR / "notebook_command_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/hntnhan1111-ayai/CVio_Shrimp_Disease_Classification_Capstone_SU26.git"
BRANCH_NAME = "feature/improving-lightweight-shrimp-disease-classification-coinfection-losses-randaugment"

RUN_GIT_PULL = True
BACKUP_NON_GIT_REPO_DIR = True
FORCE_DELETE_NON_GIT_REPO_DIR = False

def utc_stamp():
    return datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

def normalize_branch_name(branch):
    branch = str(branch).strip()
    if branch.startswith("origin/"):
        branch = branch[len("origin/"):]
    return branch

BRANCH_NAME = normalize_branch_name(BRANCH_NAME)

def run_cmd(cmd, cwd=None, log_name="command", check=True, env=None, safe_cmd=None):
    cwd = Path(cwd or WORKDIR).expanduser().resolve()
    cwd.mkdir(parents=True, exist_ok=True)

    log_path = LOG_DIR / f"{utc_stamp()}_{log_name}.log"
    shown = safe_cmd if safe_cmd is not None else cmd
    shown_text = " ".join(shlex.quote(str(x)) for x in shown)

    merged_env = os.environ.copy()
    merged_env["GIT_TERMINAL_PROMPT"] = "0"
    if env:
        merged_env.update(env)

    print("\n" + "=" * 110, flush=True)
    print("RUN:", shown_text, flush=True)
    print("CWD:", cwd, flush=True)
    print("LOG:", log_path, flush=True)
    print("=" * 110, flush=True)

    start = time.time()
    with open(log_path, "w", encoding="utf-8", errors="replace") as f:
        f.write("RUN: " + shown_text + "\n")
        f.write("CWD: " + str(cwd) + "\n\n")
        f.flush()

        proc = subprocess.Popen(
            [str(x) for x in cmd],
            cwd=str(cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=merged_env,
        )

        for line in proc.stdout:
            print(line, end="", flush=True)
            f.write(line)
            f.flush()

        rc = proc.wait()

    elapsed = time.time() - start
    print(f"\nRETURN_CODE={rc} | elapsed={elapsed:.1f}s | log={log_path}", flush=True)

    if check and rc != 0:
        raise RuntimeError(f"Command failed with return code {rc}: {shown_text}\nLog: {log_path}")

    return rc

def make_git_auth_env():
    token = os.environ.get("GITHUB_TOKEN", "").strip()

    if not token:
        token = getpass.getpass(
            "GITHUB_TOKEN is not set. Paste a token for private repo access, or press Enter for public clone: "
        ).strip()

    env = {}
    if token:
        auth_dir = WORKDIR / ".git_auth_runtime"
        auth_dir.mkdir(parents=True, exist_ok=True)

        askpass = auth_dir / "git_askpass.py"
        askpass.write_text(
            "import os, sys\n"
            "prompt = sys.argv[1].lower() if len(sys.argv) > 1 else ''\n"
            "if 'username' in prompt:\n"
            "    print(os.environ.get('GITHUB_USERNAME', 'x-access-token'))\n"
            "else:\n"
            "    print(os.environ.get('GITHUB_TOKEN', ''))\n",
            encoding="utf-8",
        )
        askpass.chmod(0o700)

        env.update({
            "GIT_ASKPASS": str(askpass),
            "GITHUB_USERNAME": os.environ.get("GITHUB_USERNAME", "x-access-token"),
            "GITHUB_TOKEN": token,
            "GIT_TERMINAL_PROMPT": "0",
        })

        print("Git token loaded via hidden prompt/environment and will not be printed.")
    else:
        print("No token provided. Clone/pull will work only if the repository is public.")

    return env

def backup_or_remove_non_git_dir(path):
    path = Path(path).expanduser().resolve()

    if not path.exists():
        return

    if (path / ".git").exists():
        return

    if not path.is_dir():
        raise RuntimeError(f"REPO_DIR exists but is not a directory: {path}")

    if FORCE_DELETE_NON_GIT_REPO_DIR:
        print(f"Deleting non-Git REPO_DIR because FORCE_DELETE_NON_GIT_REPO_DIR=True: {path}")
        shutil.rmtree(path)
        return

    if BACKUP_NON_GIT_REPO_DIR:
        backup_path = path.with_name(path.name + f"_non_git_backup_{utc_stamp()}")
        print("REPO_DIR exists but is not a Git checkout.")
        print("Moving it to backup instead of deleting:")
        print("FROM:", path)
        print("TO:  ", backup_path)
        shutil.move(str(path), str(backup_path))
        return

    raise RuntimeError(
        "REPO_DIR exists but is not a Git checkout. "
        "Set BACKUP_NON_GIT_REPO_DIR=True to move it aside, "
        "or set FORCE_DELETE_NON_GIT_REPO_DIR=True to delete it."
    )

if RUN_GIT_PULL:
    os.chdir(WORKDIR)
    WORKDIR.mkdir(parents=True, exist_ok=True)

    git_env = make_git_auth_env()

    backup_or_remove_non_git_dir(REPO_DIR)

    if not REPO_DIR.exists():
        run_cmd(
            ["git", "clone", "--branch", BRANCH_NAME, "--single-branch", REPO_URL, str(REPO_DIR)],
            cwd=WORKDIR,
            log_name="git_clone",
            env=git_env,
        )
    else:
        run_cmd(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_DIR, log_name="git_remote_plain", env=git_env)
        run_cmd(["git", "fetch", "origin", BRANCH_NAME], cwd=REPO_DIR, log_name="git_fetch", env=git_env)
        run_cmd(["git", "checkout", BRANCH_NAME], cwd=REPO_DIR, log_name="git_checkout", env=git_env)
        run_cmd(["git", "pull", "--ff-only", "origin", BRANCH_NAME], cwd=REPO_DIR, log_name="git_pull_ff_only", env=git_env)

    run_cmd(["git", "branch", "--show-current"], cwd=REPO_DIR, log_name="git_branch")
    run_cmd(["git", "status", "--short"], cwd=REPO_DIR, log_name="git_status")
    run_cmd(["git", "log", "--oneline", "-5"], cwd=REPO_DIR, log_name="git_log")
else:
    print("Skipped Git pull. Set RUN_GIT_PULL=True to pull latest code.")

os.chdir(REPO_DIR)
print("Current working directory:", Path.cwd())

Git token loaded via hidden prompt/environment and will not be printed.
REPO_DIR exists but is not a Git checkout.
Moving it to backup instead of deleting:
FROM: /home/drnguyenvinh/notebooks/CVio_Shrimp_Disease_Classification_Capstone_SU26
TO:   /home/drnguyenvinh/notebooks/CVio_Shrimp_Disease_Classification_Capstone_SU26_non_git_backup_20260601_013347

RUN: git clone --branch feature/improving-lightweight-shrimp-disease-classification-coinfection-losses-randaugment --single-branch https://github.com/hntnhan1111-ayai/CVio_Shrimp_Disease_Classification_Capstone_SU26.git /home/drnguyenvinh/notebooks/CVio_Shrimp_Disease_Classification_Capstone_SU26
CWD: /home/drnguyenvinh/notebooks
LOG: /home/drnguyenvinh/notebooks/notebook_command_logs/20260601_013347_git_clone.log
Cloning into '/home/drnguyenvinh/notebooks/CVio_Shrimp_Disease_Classification_Capstone_SU26'...

RETURN_CODE=0 | elapsed=1.3s | log=/home/drnguyenvinh/notebooks/notebook_command_logs/20260601_013347_git_clone.log

RUN: git bra

In [2]:
LOG_DIR.mkdir(parents=True, exist_ok=True)

def utc_stamp() -> str:
    return datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

def run_cmd(cmd, cwd=None, log_name="command", check=True, env=None, safe_cmd=None):
    cwd = Path(cwd or WORKDIR).expanduser().resolve()
    cwd.mkdir(parents=True, exist_ok=True)
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    log_path = LOG_DIR / f"{utc_stamp()}_{log_name}.log"
    shown = safe_cmd if safe_cmd is not None else cmd
    shown_text = " ".join(shlex.quote(str(x)) for x in shown)
    merged_env = os.environ.copy()
    merged_env["PYTHONUNBUFFERED"] = "1"
    merged_env["GIT_TERMINAL_PROMPT"] = "0"
    if env:
        merged_env.update({str(k): str(v) for k, v in env.items()})
    print("\n" + "=" * 110, flush=True)
    print("RUN:", shown_text, flush=True)
    print("CWD:", cwd, flush=True)
    print("LOG:", log_path, flush=True)
    print("=" * 110, flush=True)
    start = time.time()
    with open(log_path, "w", encoding="utf-8", errors="replace") as f:
        f.write("RUN: " + shown_text + "\n")
        f.write("CWD: " + str(cwd) + "\n\n")
        f.flush()
        proc = subprocess.Popen(
            [str(x) for x in cmd],
            cwd=str(cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=merged_env,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
            f.write(line)
            f.flush()
        rc = proc.wait()
    elapsed = time.time() - start
    print(f"\nRETURN_CODE={rc} | elapsed={elapsed:.1f}s | log={log_path}", flush=True)
    if check and rc != 0:
        raise RuntimeError(f"Command failed with return code {rc}: {shown_text}\nLog: {log_path}")
    return rc, log_path

def py_script(script, *args, cwd=None, log_name=None, check=True, env=None):
    return run_cmd([sys.executable, "-u", script, *[str(a) for a in args]], cwd=cwd or REPO_DIR, log_name=log_name or Path(script).stem, check=check, env=env)

print("Helpers loaded.")

Helpers loaded.


In [3]:
def make_git_auth_env():
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    if not token:
        token = getpass.getpass("GITHUB_TOKEN is not set. Paste a token for private repo access, or press Enter for public clone: ").strip()
    env = {}
    if token:
        auth_dir = WORKDIR / ".git_auth_runtime"
        auth_dir.mkdir(parents=True, exist_ok=True)
        askpass = auth_dir / "git_askpass.py"
        askpass.write_text(
            "import os, sys\n"
            "prompt = sys.argv[1].lower() if len(sys.argv) > 1 else ''\n"
            "if 'username' in prompt:\n"
            "    print(os.environ.get('GITHUB_USERNAME', 'x-access-token'))\n"
            "else:\n"
            "    print(os.environ.get('GITHUB_TOKEN', ''))\n",
            encoding="utf-8",
        )
        askpass.chmod(0o700)
        env.update({
            "GIT_ASKPASS": str(askpass),
            "GITHUB_USERNAME": os.environ.get("GITHUB_USERNAME", "x-access-token"),
            "GITHUB_TOKEN": token,
            "GIT_TERMINAL_PROMPT": "0",
        })
        print("Git token loaded via hidden prompt/environment and will not be printed.")
    else:
        print("No token provided. Clone/pull will work only if the repository is public.")
    return env

if RUN_GIT_PULL:
    git_env = make_git_auth_env()
    WORKDIR.mkdir(parents=True, exist_ok=True)
    if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
        raise RuntimeError(f"REPO_DIR exists but is not a Git checkout: {REPO_DIR}")
    if not REPO_DIR.exists():
        run_cmd(
            ["git", "clone", "--branch", BRANCH_NAME, "--single-branch", REPO_URL, str(REPO_DIR)],
            cwd=WORKDIR,
            log_name="git_clone",
            env=git_env,
        )
    else:
        run_cmd(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_DIR, log_name="git_remote_plain", env=git_env)
        run_cmd(["git", "fetch", "origin", BRANCH_NAME], cwd=REPO_DIR, log_name="git_fetch", env=git_env)
        run_cmd(["git", "checkout", BRANCH_NAME], cwd=REPO_DIR, log_name="git_checkout", env=git_env)
        run_cmd(["git", "pull", "--ff-only", "origin", BRANCH_NAME], cwd=REPO_DIR, log_name="git_pull_ff_only", env=git_env)
    run_cmd(["git", "branch", "--show-current"], cwd=REPO_DIR, log_name="git_branch")
    run_cmd(["git", "status", "--short"], cwd=REPO_DIR, log_name="git_status")
    run_cmd(["git", "log", "--oneline", "-5"], cwd=REPO_DIR, log_name="git_log")
else:
    print("Skipped Git pull. Set RUN_GIT_PULL=True to pull latest code.")

os.chdir(REPO_DIR)
print("Current working directory:", Path.cwd())

Git token loaded via hidden prompt/environment and will not be printed.


RuntimeError: REPO_DIR exists but is not a Git checkout: /home/drnguyenvinh/notebooks/CVio_Shrimp_Disease_Classification_Capstone_SU26

In [ ]:
import importlib.util

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "Pillow",
    "sklearn": "scikit-learn",
    "torch": "torch",
    "torchvision": "torchvision",
    "timm": "timm",
    "ultralytics": "ultralytics",
    "tqdm": "tqdm",
    "matplotlib": "matplotlib",
    "openpyxl": "openpyxl",
    "yaml": "PyYAML",
    "cv2": "opencv-python",
}

missing = []
for module_name, package_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        missing.append((module_name, package_name))

print("Missing modules:", missing)

if RUN_INSTALL_MISSING_PACKAGES and missing:
    packages = []
    for module_name, package_name in missing:
        if module_name in {"torch", "torchvision"} and not INSTALL_TORCH_IF_MISSING:
            raise RuntimeError("torch/torchvision missing. Install the correct CUDA build manually or set INSTALL_TORCH_IF_MISSING=True.")
        packages.append(package_name)
    unique_packages = []
    for pkg in packages:
        if pkg not in unique_packages:
            unique_packages.append(pkg)
    run_cmd([sys.executable, "-m", "pip", "install", *unique_packages], cwd=REPO_DIR, log_name="pip_install_missing")
else:
    print("No missing packages or installation disabled.")

import torch
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("cuda_device_count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

In [ ]:
def resolve_local_dataset_root(dataset_input: str | Path) -> Path:
    raw = Path(dataset_input).expanduser()
    candidates = []
    if raw.is_absolute():
        candidates.append(raw)
    else:
        candidates.extend([
            (Path.cwd() / raw),
            (REPO_DIR / raw),
            (WORKDIR / raw),
            (REPO_DIR.parent / raw),
        ])
    more = []
    for c in candidates:
        more.extend([c, c / "processed-images", c / "processed_images", c / "processed-images" / "processed-images", c / "processed_images" / "processed_images"])
    class_dirs = ["1. Healthy", "2. BG", "3. WSSV", "4. WSSV_BG"]
    attempts = []
    for c in more:
        r = c.resolve()
        ok = r.is_dir() and all((r / d).is_dir() for d in class_dirs)
        attempts.append({"candidate": str(r), "ok": ok})
        if ok:
            return r
    print(json.dumps({"dataset_resolution_attempts": attempts}, indent=2))
    raise FileNotFoundError("Could not find local dataset root containing class folders: " + ", ".join(class_dirs))

DATASET_ROOT = resolve_local_dataset_root(DATASET_ROOT_INPUT)
print("DATASET_ROOT:", DATASET_ROOT)

runtime_overrides = REPO_DIR / "_local_runtime_overrides"
runtime_overrides.mkdir(parents=True, exist_ok=True)
(runtime_overrides / "kagglehub.py").write_text(
    "from pathlib import Path\n"
    "def dataset_download(dataset_id):\n"
    "    return r'''%s'''\n" % str(DATASET_ROOT),
    encoding="utf-8",
)

SCRIPT_ENV = os.environ.copy()
SCRIPT_ENV["PYTHONPATH"] = str(runtime_overrides) + os.pathsep + str(REPO_DIR) + os.pathsep + SCRIPT_ENV.get("PYTHONPATH", "")
SCRIPT_ENV["SHRIMP_DATASET_ROOT"] = str(DATASET_ROOT)
SCRIPT_ENV["PYTHONUNBUFFERED"] = "1"

print(json.dumps({
    "dataset_root": str(DATASET_ROOT),
    "runtime_override_kagglehub": str(runtime_overrides / "kagglehub.py"),
    "pythonpath_prefix": [str(runtime_overrides), str(REPO_DIR)],
}, indent=2))

In [ ]:
if RUN_STATIC_CHECKS:
    py_script("-m", "compileall", "shrimp_scripts", "experiments", cwd=REPO_DIR, log_name="compileall", env=SCRIPT_ENV)
    scan_targets = list((REPO_DIR / "shrimp_scripts").glob("*.py")) + list((REPO_DIR / "experiments").rglob("*.py"))
    violations = []
    for path in scan_targets:
        text = path.read_text(encoding="utf-8", errors="replace")
        bad_terms = []
        for term in ["task=segment", "yolo26m-seg", "ExecuTorch", "NCNN", "TFLite"]:
            if term in text:
                bad_terms.append(term)
        if bad_terms:
            violations.append({"file": str(path.relative_to(REPO_DIR)), "terms": bad_terms})
    print(json.dumps({"static_scope_violations": violations}, indent=2))
    if violations:
        raise RuntimeError("Unexpected segmentation/export terms found in training scripts.")
else:
    print("Skipped static checks.")

In [ ]:
if RUN_PREPARE_DATASET:
    py_script(
        "shrimp_scripts/run_01_prepare_dataset.py",
        "--output_dir", str(OUTPUT_DIR_MAIN),
        "--resume",
        "--progress",
        cwd=REPO_DIR,
        log_name="run_01_prepare_dataset",
        env=SCRIPT_ENV,
    )
else:
    print("Skipped dataset preparation.")

In [ ]:
if RUN_LIST_RUNS:
    py_script("shrimp_scripts/run_02_train_core_ablation.py", "--output_dir", str(OUTPUT_DIR_MAIN), "--list_runs", cwd=REPO_DIR, log_name="list_stage02", env=SCRIPT_ENV)
    py_script("shrimp_scripts/run_03_train_yolo_family.py", "--output_dir", str(OUTPUT_DIR_MAIN), "--list_runs", "--skip_probe", cwd=REPO_DIR, log_name="list_stage03", env=SCRIPT_ENV)
    py_script("shrimp_scripts/run_04_train_lightweight_models.py", "--output_dir", str(OUTPUT_DIR_MAIN), "--list_runs", cwd=REPO_DIR, log_name="list_stage04", env=SCRIPT_ENV)
    py_script("experiments/asl_custom_loss_screening/run_asl_custom_screen.py", "--output_dir", str(OUTPUT_DIR_ASL), "--list_runs", cwd=REPO_DIR, log_name="list_asl_screening", env=SCRIPT_ENV)
else:
    print("Skipped run listing.")

In [ ]:
if RUN_VALIDATE_RESUME_BEFORE:
    py_script("shrimp_scripts/run_02_train_core_ablation.py", "--output_dir", str(OUTPUT_DIR_MAIN), "--validate_resume", cwd=REPO_DIR, log_name="validate_resume_stage02_before", env=SCRIPT_ENV)
    py_script("shrimp_scripts/run_03_train_yolo_family.py", "--output_dir", str(OUTPUT_DIR_MAIN), "--validate_resume", "--skip_probe", cwd=REPO_DIR, log_name="validate_resume_stage03_before", env=SCRIPT_ENV)
    py_script("shrimp_scripts/run_04_train_lightweight_models.py", "--output_dir", str(OUTPUT_DIR_MAIN), "--validate_resume", cwd=REPO_DIR, log_name="validate_resume_stage04_before", env=SCRIPT_ENV)
    py_script("experiments/asl_custom_loss_screening/run_asl_custom_screen.py", "--output_dir", str(OUTPUT_DIR_ASL), "--validate_resume", cwd=REPO_DIR, log_name="validate_resume_asl_before", env=SCRIPT_ENV)
else:
    print("Skipped pre-run resume validation.")

In [ ]:
if RUN_STAGE_02_CORE_ABLATION:
    py_script(
        "shrimp_scripts/run_02_train_core_ablation.py",
        "--output_dir", str(OUTPUT_DIR_MAIN),
        "--resume",
        "--progress",
        cwd=REPO_DIR,
        log_name="stage02_core_ablation",
        env=SCRIPT_ENV,
    )
else:
    print("Skipped Stage 02.")

In [ ]:
if RUN_STAGE_03_YOLO_FAMILY:
    py_script(
        "shrimp_scripts/run_03_train_yolo_family.py",
        "--output_dir", str(OUTPUT_DIR_MAIN),
        "--resume",
        "--progress",
        cwd=REPO_DIR,
        log_name="stage03_yolo_family",
        env=SCRIPT_ENV,
    )
else:
    print("Skipped Stage 03.")

In [ ]:
if RUN_STAGE_04_LIGHTWEIGHT_CHUNK_0:
    py_script(
        "shrimp_scripts/run_04_train_lightweight_models.py",
        "--output_dir", str(OUTPUT_DIR_MAIN),
        "--resume",
        "--progress",
        "--start", "0",
        "--limit", "17",
        cwd=REPO_DIR,
        log_name="stage04_lightweight_chunk_0_17",
        env=SCRIPT_ENV,
    )
else:
    print("Skipped Stage 04 chunk 0.")

if RUN_STAGE_04_LIGHTWEIGHT_CHUNK_1:
    py_script(
        "shrimp_scripts/run_04_train_lightweight_models.py",
        "--output_dir", str(OUTPUT_DIR_MAIN),
        "--resume",
        "--progress",
        "--start", "17",
        "--limit", "17",
        cwd=REPO_DIR,
        log_name="stage04_lightweight_chunk_17_34",
        env=SCRIPT_ENV,
    )
else:
    print("Skipped Stage 04 chunk 1.")

if RUN_STAGE_04_LIGHTWEIGHT_CHUNK_2:
    py_script(
        "shrimp_scripts/run_04_train_lightweight_models.py",
        "--output_dir", str(OUTPUT_DIR_MAIN),
        "--resume",
        "--progress",
        "--start", "34",
        "--limit", "17",
        cwd=REPO_DIR,
        log_name="stage04_lightweight_chunk_34_51",
        env=SCRIPT_ENV,
    )
else:
    print("Skipped Stage 04 chunk 2.")

In [ ]:
if RUN_STAGE_05_REPORTS_XAI:
    py_script(
        "shrimp_scripts/run_05_generate_reports_and_xai.py",
        "--output_dir", str(OUTPUT_DIR_MAIN),
        "--resume",
        "--progress",
        cwd=REPO_DIR,
        log_name="stage05_reports_xai",
        env=SCRIPT_ENV,
    )
else:
    print("Skipped Stage 05.")

In [ ]:
if RUN_ASL_CUSTOM_SCREENING:
    py_script(
        "experiments/asl_custom_loss_screening/run_asl_custom_screen.py",
        "--output_dir", str(OUTPUT_DIR_ASL),
        "--backend", "all",
        "--resume",
        "--progress",
        cwd=REPO_DIR,
        log_name="asl_custom_loss_screening_all",
        env=SCRIPT_ENV,
    )
    py_script(
        "experiments/asl_custom_loss_screening/collect_asl_custom_results.py",
        "--output_dir", str(OUTPUT_DIR_ASL),
        cwd=REPO_DIR,
        log_name="collect_asl_custom_results",
        env=SCRIPT_ENV,
    )
else:
    print("Skipped ASL custom-loss screening.")

In [ ]:
if RUN_VALIDATE_RESUME_AFTER:
    py_script("shrimp_scripts/run_02_train_core_ablation.py", "--output_dir", str(OUTPUT_DIR_MAIN), "--validate_resume", cwd=REPO_DIR, log_name="validate_resume_stage02_after", env=SCRIPT_ENV)
    py_script("shrimp_scripts/run_03_train_yolo_family.py", "--output_dir", str(OUTPUT_DIR_MAIN), "--validate_resume", "--skip_probe", cwd=REPO_DIR, log_name="validate_resume_stage03_after", env=SCRIPT_ENV)
    py_script("shrimp_scripts/run_04_train_lightweight_models.py", "--output_dir", str(OUTPUT_DIR_MAIN), "--validate_resume", cwd=REPO_DIR, log_name="validate_resume_stage04_after", env=SCRIPT_ENV)
    py_script("experiments/asl_custom_loss_screening/run_asl_custom_screen.py", "--output_dir", str(OUTPUT_DIR_ASL), "--validate_resume", cwd=REPO_DIR, log_name="validate_resume_asl_after", env=SCRIPT_ENV)
else:
    print("Skipped post-run resume validation.")

In [ ]:
import pandas as pd

def read_json_safe(path):
    path = Path(path)
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        return {"read_error": repr(exc)}

def collect_status(output_dir):
    output_dir = Path(output_dir)
    rows = []
    for status_path in sorted((output_dir / "runs").glob("*/status.json")):
        run_dir = status_path.parent
        status = read_json_safe(status_path)
        audit = read_json_safe(run_dir / "run_audit.json")
        metrics = read_json_safe(run_dir / "metrics.json")
        class_order = read_json_safe(run_dir / "class_order_audit.json")
        row = {
            "run_dir": run_dir.name,
            "status": status.get("status"),
            "backend": metrics.get("backend") or audit.get("backend") or metrics.get("screen_backend"),
            "model": metrics.get("model") or audit.get("model_name") or audit.get("model_key"),
            "loss_key": metrics.get("loss_key") or audit.get("loss_key"),
            "condition": metrics.get("condition") or audit.get("condition_key") or (audit.get("condition") or {}).get("condition_key"),
            "randaugment": metrics.get("randaugment") if "randaugment" in metrics else audit.get("randaugment"),
            "test_accuracy": metrics.get("test_accuracy") or (metrics.get("test") or {}).get("accuracy"),
            "test_macro_f1": metrics.get("test_macro_f1") or (metrics.get("test") or {}).get("macro_f1"),
            "cohen_kappa": metrics.get("cohen_kappa") or (metrics.get("test") or {}).get("cohen_kappa"),
            "class_order_audit_passed": class_order.get("audit_passed"),
            "swap_delta": (class_order.get("swap_diagnostic") or {}).get("macro_f1_delta_after_swap"),
            "error": status.get("error") or status.get("exception") or status.get("read_error"),
        }
        rows.append(row)
    return pd.DataFrame(rows)

if RUN_COLLECT_STATUS:
    for label, out in [("main", OUTPUT_DIR_MAIN), ("asl", OUTPUT_DIR_ASL)]:
        df = collect_status(out)
        print("\n" + "=" * 100)
        print(label, out, "rows:", len(df))
        if not df.empty:
            display(df.sort_values(["status", "run_dir"], na_position="last"))
            display(df["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="count"))
else:
    print("Skipped status collection.")

In [ ]:
if RUN_BACKUP_OUTPUTS:
    import zipfile
    backup_path = REPO_DIR / f"shrimp_outputs_rtx4090_backup_{utc_stamp()}.zip"
    include_roots = [OUTPUT_DIR_MAIN, OUTPUT_DIR_ASL, LOG_DIR]
    with zipfile.ZipFile(backup_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for root in include_roots:
            root = Path(root)
            if not root.exists():
                continue
            for path in root.rglob("*"):
                if not path.is_file():
                    continue
                if path.suffix.lower() in {".pt", ".pth", ".onnx", ".engine", ".tflite"}:
                    continue
                try:
                    zf.write(path, arcname=str(path.relative_to(REPO_DIR)))
                except ValueError:
                    zf.write(path, arcname=path.name)
    print("Backup created:", backup_path)
else:
    print("Skipped backup.")